# Extraction OEDI — timeseries & météo statique
Deux extracteurs depuis le data lake public OEDI (ResStock 2025, AMY2018) :
- **Partie A** — timeseries par bâtiment (35 040 pas de 15 min × 192 colonnes).
- **Partie B** — météo statique par comté (agrégats climatiques : HDD, CDD, GHI, T design, vent, humidité), joignables sur `in.county`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_RAW       = ROOT / 'data' / 'raw'

# Partie A — timeseries par bâtiment
OEDI_BASE = (
    'https://oedi-data-lake.s3.amazonaws.com/'
    'nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/'
    '2025/resstock_amy2018_release_1/'
    'timeseries_individual_buildings/by_state/upgrade=0'
)

# Partie B — météo par comté (même release)
WEATHER_BASE = (
    'https://oedi-data-lake.s3.amazonaws.com/'
    'nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/'
    '2025/resstock_amy2018_release_1/weather'
)

## 1bis. Reconstruction des consignes de thermostat

Les consignes (chauffage + clim) sont des schedules **non stochastiques** : absentes (colonnes
nulles) du time series, mais entièrement déterministes à partir de 4 caractéristiques statiques
(`metadata_clean.parquet`) + le masque horaire de `options_lookup.tsv` :

$$\text{consigne}(h) = \text{base} + \text{masque}(h) \times \text{magnitude}$$

`inject_setpoints(df, bldg_id)` ajoute deux colonnes en °C. Elle est appelée automatiquement
à chaque téléchargement (fonctions ci-dessous).

In [ ]:
import re, functools

META_PATH   = DATA_PROCESSED / 'metadata_clean.parquet'
LOOKUP_PATH = ROOT / 'data' / 'external' / 'options_lookup.tsv'   # depot NREL/resstock
COL_COOL    = 'out.schedules.cooling_setpoint..c'
COL_HEAT    = 'out.schedules.heating_setpoint..c'

def _f_abs_to_c(txt):    # '76F' -> 24.44  (temperature absolue)
    return (float(re.search(r'[-\d.]+', str(txt)).group()) - 32) * 5 / 9

def _f_delta_to_c(txt):  # '9F' -> 5.0  (ecart : PAS de -32)
    return float(re.search(r'[-\d.]+', str(txt)).group()) * 5 / 9

@functools.lru_cache(maxsize=1)
def _lookup_lines():
    return LOOKUP_PATH.read_text(encoding='utf-8').splitlines()

@functools.lru_cache(maxsize=None)
def _get_masks(characteristic, period):
    """(masque_semaine[24], masque_weekend[24]) signes -1/0/+1."""
    if period is None or str(period).strip().lower() == 'none':
        return tuple(np.zeros(24)), tuple(np.zeros(24))
    prefix = f'{characteristic}\t{period}\t'
    line = next((l for l in _lookup_lines() if l.startswith(prefix)), None)
    if line is None:
        raise ValueError(f'"{characteristic}" / "{period}" introuvable dans options_lookup')
    wk = re.search(r'weekday_setpoint_schedule=([-\d,\s]+)', line).group(1)
    we = re.search(r'weekend_setpoint_schedule=([-\d,\s]+)', line).group(1)
    arr = lambda s: tuple(int(x) for x in s.split(',')[:24])
    return arr(wk), arr(we)

@functools.lru_cache(maxsize=1)
def _setpoint_meta():
    cols = ['bldg_id',
            'in.cooling_setpoint', 'in.cooling_setpoint_has_offset',
            'in.cooling_setpoint_offset_magnitude', 'in.cooling_setpoint_offset_period',
            'in.heating_setpoint', 'in.heating_setpoint_has_offset',
            'in.heating_setpoint_offset_magnitude', 'in.heating_setpoint_offset_period']
    return pd.read_parquet(META_PATH, columns=cols).set_index('bldg_id')

def inject_setpoints(df, bldg_id):
    """Reconstruit et injecte les consignes chauffage + clim (colonnes en degC)."""
    ts = pd.DatetimeIndex(df['timestamp'] if 'timestamp' in df.columns else df.index)
    hours, wknd = ts.hour.values, (ts.dayofweek.values >= 5)
    row = _setpoint_meta().loc[bldg_id]
    for col, pfx, param in [
            (COL_COOL, 'in.cooling_setpoint', 'Cooling Setpoint Offset Period'),
            (COL_HEAT, 'in.heating_setpoint', 'Heating Setpoint Offset Period')]:
        has  = row[f'{pfx}_has_offset'] == 'Yes'
        base = _f_abs_to_c(row[pfx])
        mag  = _f_delta_to_c(row[f'{pfx}_offset_magnitude']) if has else 0.0
        wk, we = _get_masks(param, row[f'{pfx}_offset_period'] if has else None)
        m = np.where(wknd, np.asarray(we, float)[hours], np.asarray(wk, float)[hours])
        df[col] = base + m * mag
    return df


# Partie A — Timeseries individuelles par bâtiment

## 1. Filtrer les bâtiments (filtre du modèle)
Maison individuelle plein-pied, chauffage électrique, **sans VE / piscine / PV**, et **occupé**
(les vacants ont des schedules vides). Même périmètre que `lgbm_electricity_5features`.

In [ ]:
raw = pd.read_parquet(DATA_RAW / 'upgrade0.parquet', columns=[
    'bldg_id', 'in.state', 'in.county', 'in.ashrae_iecc_climate_zone_2004',
    'in.geometry_building_type_recs', 'in.geometry_stories', 'in.heating_fuel',
    'in.electric_vehicle_ownership', 'in.misc_pool', 'in.has_pv', 'in.vacancy_status',
])

mask = (
    (raw['in.geometry_building_type_recs'] == 'Single-Family Detached') &
    (raw['in.geometry_stories'] == '1')                                  &
    (raw['in.heating_fuel'] == 'Electricity')                            &
    (raw['in.electric_vehicle_ownership'] == 'No')                       &
    (raw['in.misc_pool'] == 'None')                                      &
    (raw['in.has_pv'] == 'No')                                           &
    (raw['in.vacancy_status'] == 'Occupied')
)

candidates = raw[mask][['bldg_id', 'in.state', 'in.county',
                        'in.ashrae_iecc_climate_zone_2004']].reset_index(drop=True)
print(f'{len(candidates):,} bâtiments (filtre modèle + occupé)')
candidates.head(10)

## 2. Télécharger la timeserie d'un bâtiment
Les fichiers sont publics sur OEDI — aucune clé AWS requise.  
URL : `{OEDI_BASE}/state={state}/{bldg_id}-0.parquet`

In [ ]:
def download_timeseries(bldg_id: int, state: str, save: bool = True) -> pd.DataFrame:
    url = f'{OEDI_BASE}/state={state}/{bldg_id}-0.parquet'
    print(f'Téléchargement : {url}')
    df = pd.read_parquet(url)
    df = inject_setpoints(df, bldg_id)          # consignes reconstruites (chauffage + clim)
    if save:
        out = DATA_PROCESSED / f'{bldg_id}-0.parquet'
        df.to_parquet(out)
        print(f'Sauvegardé : {out}')
    print(f'Shape : {df.shape} | {df["timestamp"].iloc[0]} → {df["timestamp"].iloc[-1]}')
    return df

In [ ]:
# Exemple : le premier bâtiment candidat
row   = candidates.iloc[0]
df_ts = download_timeseries(int(row['bldg_id']), row['in.state'])
df_ts.head()

## 3. Télécharger un lot pour le réseau (échantillon stratifié par zone)
Lecture **par colonnes** (météo + schedules + cibles) → ~1 Mo/bâtiment au lieu de 9,3.
Échantillon réparti sur les zones climatiques.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# Colonnes utiles au réseau (parquet colonnaire -> ~1 Mo/bâtiment)
WEA = ['out.outdoor_air_drybulb_temp..c', 'out.outdoor_air_relative_humidity..percentage',
       'out.weather.wind_speed..meter_per_second',
       'out.weather.direct_normal_solar_radiation..watt_per_m2',
       'out.weather.diffuse_solar_radiation..watt_per_m2']
SCHED = ['out.schedules.' + s for s in [
    'occupants', 'vacancy', 'lighting_interior', 'lighting_garage', 'plug_loads_other',
    'plug_loads_tv', 'clothes_dryer', 'clothes_washer', 'dishwasher', 'cooking_range',
    'ceiling_fan', 'hot_water_fixtures', 'hot_water_clothes_washer', 'hot_water_dishwasher',
    'no_space_cooling', 'no_space_heating']]
TGT = ['out.electricity.' + t + '.energy_consumption..kwh' for t in
       ['total', 'heating', 'cooling', 'hot_water']]
NN_COLS = ['timestamp'] + WEA + SCHED + TGT

def download_nn(bldg_id, state):
    out = DATA_PROCESSED / f'{bldg_id}-0.parquet'
    if out.exists():
        return
    df = pd.read_parquet(f'{OEDI_BASE}/state={state}/{bldg_id}-0.parquet', columns=NN_COLS)
    df = inject_setpoints(df, bldg_id)          # consignes reconstruites (chauffage + clim)
    df.to_parquet(out)

# Échantillon stratifié par zone climatique (>= 1 par zone, ~N au total)
N, zcol = 150, 'in.ashrae_iecc_climate_zone_2004'
parts = [g.sample(min(len(g), max(1, round(N * len(g) / len(candidates)))), random_state=42)
         for _, g in candidates.groupby(zcol)]
sample = pd.concat(parts).reset_index(drop=True)
print(f'{len(sample)} bâtiments à télécharger sur {sample[zcol].nunique()} zones')

# Téléchargement parallèle (8 en simultané -> ~1-2 min)
jobs = list(zip(sample['bldg_id'].astype(int), sample['in.state']))
def _get(job):
    bid, st = job
    try:
        download_nn(bid, st)
    except Exception as e:
        return f'  skip {bid} : {e}'

with ThreadPoolExecutor(max_workers=8) as ex:
    for msg in ex.map(_get, jobs):
        if msg:
            print(msg)

sample[['bldg_id', 'in.state']].to_csv(DATA_PROCESSED / 'nn_buildings.csv', index=False)
print('terminé ->', DATA_PROCESSED / 'nn_buildings.csv')

# Partie B — Météo statique par comté

## 4. Agrégats climatiques
Un CSV météo léger par comté (= la météo exacte de simulation, validée : écart ~0). On en extrait des grandeurs statiques qui couplent avec les agrégats d'enveloppe — `HDD/CDD` × `UA`, `GHI` × `A_solaire`, `vent` × `H_ve`. Jointure sur `in.county`.  
URL : `{WEATHER_BASE}/state={state}/{county}_2018.csv`

In [ ]:
def weather_static(county: str, state: str, base: float = 18.0) -> dict:
    """Agrégats climatiques annuels d'un comté (CSV météo horaire OEDI)."""
    w = pd.read_csv(f'{WEATHER_BASE}/state={state}/{county}_2018.csv', parse_dates=['date_time'])
    T, ghi = w['Dry Bulb Temperature [°C]'], w['Global Horizontal Radiation [W/m2]']
    hiver = w['date_time'].dt.month.isin([12, 1, 2])
    return {
        'in.county'   : county,
        'HDD18'       : np.maximum(base - T, 0).sum() / 24,   # °C·jour chauffage
        'CDD18'       : np.maximum(T - base, 0).sum() / 24,   # °C·jour clim
        'T_moy'       : T.mean(),
        'T_design_min': T.quantile(0.01),
        'T_design_max': T.quantile(0.99),
        'GHI_an'      : ghi.sum() / 1000,                     # kWh/m²/an
        'GHI_hiver'   : ghi[hiver].sum() / 1000,
        'vent_moy'    : w['Wind Speed [m/s]'].mean(),
        'RH_moy'      : w['Relative Humidity [%]'].mean(),
    }

# test sur le comté du bâtiment 159
weather_static('G4804010', 'TX')

In [ ]:
# Batch : 1 fichier météo par comté (partagé par tous ses bâtiments) -> weather_static.parquet
counties = (candidates[['in.county', 'in.state']]
            .drop_duplicates()
            .loc[lambda d: d['in.county'].str.startswith('G')])   # exclut AK/HI (non-GISJOIN)

rows = []
for _, r in counties.iterrows():
    try:
        rows.append(weather_static(r['in.county'], r['in.state']))
    except Exception as e:
        print(f'  skip {r["in.county"]} : {e}')

weather = pd.DataFrame(rows)
weather.to_parquet(DATA_PROCESSED / 'weather_static.parquet')
print(f'{len(weather)} comtés → weather_static.parquet')
weather.head()